# PCA on the Pokemon stats — notebook version

This notebook aggregates the project into one runnable file. Each section corresponds to one of the `.py` modules in the project folder. Run the cells top to bottom.

**Requirements:** `pandas`, `scikit-learn`, `plotly`.

In [ ]:
# Uncomment the next line if the libraries are missing.
# %pip install pandas scikit-learn plotly

from typing import Iterable, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## 1. Non-final evolutions lookup

A hand-curated set of every Pokemon (Gen 1-6) that has a further evolution within that range. Anything **not** in this set is a final evolution.

In [ ]:
NON_FINALS: set[str] = {
    # ============================================================
    # Gen 1 — 81 pre-evolutions
    # ============================================================
    "Bulbasaur", "Ivysaur", "Charmander", "Charmeleon", "Squirtle",
    "Wartortle", "Caterpie", "Metapod", "Weedle", "Kakuna", "Pidgey",
    "Pidgeotto", "Rattata", "Spearow", "Ekans", "Pikachu", "Sandshrew",
    "Nidoran♀", "NidoranF", "Nidoran F", "Nidorina",
    "Nidoran♂", "NidoranM", "Nidoran M", "Nidorino",
    "Clefairy", "Vulpix", "Jigglypuff", "Zubat", "Golbat", "Oddish",
    "Gloom", "Paras", "Venonat", "Diglett", "Meowth", "Psyduck",
    "Mankey", "Growlithe", "Poliwag", "Poliwhirl", "Abra", "Kadabra",
    "Machop", "Machoke", "Bellsprout", "Weepinbell", "Tentacool",
    "Geodude", "Graveler", "Ponyta", "Slowpoke", "Magnemite",
    "Magneton", "Doduo", "Seel", "Grimer", "Shellder", "Gastly",
    "Haunter", "Onix", "Drowzee", "Krabby", "Voltorb", "Exeggcute",
    "Cubone", "Lickitung", "Koffing", "Rhyhorn", "Rhydon", "Chansey",
    "Tangela", "Horsea", "Seadra", "Goldeen", "Staryu", "Scyther",
    "Electabuzz", "Magmar", "Magikarp", "Eevee", "Porygon", "Omanyte",
    "Kabuto", "Dratini", "Dragonair",

    # ============================================================
    # Gen 2 — pre-evolutions including baby forms
    # ============================================================
    "Chikorita", "Bayleef", "Cyndaquil", "Quilava", "Totodile",
    "Croconaw", "Sentret", "Hoothoot", "Ledyba", "Spinarak", "Chinchou",
    "Pichu", "Cleffa", "Igglybuff", "Togepi", "Togetic", "Natu",
    "Mareep", "Flaaffy", "Marill", "Hoppip", "Skiploom", "Sunkern",
    "Yanma", "Wooper", "Murkrow", "Misdreavus", "Pineco", "Gligar",
    "Snubbull", "Teddiursa", "Slugma", "Swinub", "Piloswine",
    "Remoraid", "Houndour", "Phanpy", "Porygon2", "Tyrogue", "Smoochum",
    "Elekid", "Magby", "Larvitar", "Pupitar", "Sneasel", "Aipom",

    # ============================================================
    # Gen 3 — pre-evolutions
    # ============================================================
    "Treecko", "Grovyle", "Torchic", "Combusken", "Mudkip", "Marshtomp",
    "Poochyena", "Zigzagoon", "Wurmple", "Silcoon", "Cascoon", "Taillow",
    "Wingull", "Ralts", "Kirlia", "Surskit", "Shroomish", "Slakoth",
    "Vigoroth", "Nincada", "Whismur", "Loudred", "Makuhita", "Nosepass",
    "Skitty", "Aron", "Lairon", "Meditite", "Electrike", "Gulpin",
    "Carvanha", "Wailmer", "Numel", "Spoink", "Trapinch", "Vibrava",
    "Cacnea", "Swablu", "Barboach", "Corphish", "Lileep", "Anorith",
    "Feebas", "Shuppet", "Duskull", "Dusclops", "Snorunt", "Spheal",
    "Sealeo", "Clamperl", "Bagon", "Shelgon", "Beldum", "Metang",
    "Roselia", "Azurill", "Wynaut",

    # ============================================================
    # Gen 4 — pre-evolutions including new baby forms
    # ============================================================
    "Turtwig", "Grotle", "Chimchar", "Monferno", "Piplup", "Prinplup",
    "Starly", "Staravia", "Bidoof", "Kricketot", "Shinx", "Luxio",
    "Cranidos", "Shieldon", "Burmy", "Combee", "Buizel", "Cherubi",
    "Shellos", "Buneary", "Glameow", "Stunky", "Bronzor", "Bonsly",
    "Happiny", "Chingling", "Gible", "Gabite", "Riolu", "Hippopotas",
    "Skorupi", "Croagunk", "Finneon", "Snover", "Mime Jr.", "Munchlax",
    "Mantyke", "Budew",

    # ============================================================
    # Gen 5 — pre-evolutions
    # ============================================================
    "Snivy", "Servine", "Tepig", "Pignite", "Oshawott", "Dewott",
    "Patrat", "Lillipup", "Herdier", "Purrloin", "Pansage", "Pansear",
    "Panpour", "Munna", "Pidove", "Tranquill", "Blitzle", "Roggenrola",
    "Boldore", "Woobat", "Drilbur", "Timburr", "Gurdurr", "Tympole",
    "Palpitoad", "Sewaddle", "Swadloon", "Venipede", "Whirlipede",
    "Cottonee", "Petilil", "Sandile", "Krokorok", "Darumaka", "Dwebble",
    "Scraggy", "Yamask", "Trubbish", "Zorua", "Minccino", "Gothita",
    "Gothorita", "Solosis", "Duosion", "Ducklett", "Vanillite",
    "Vanillish", "Deerling", "Karrablast", "Foongus", "Frillish",
    "Joltik", "Ferroseed", "Klink", "Klang", "Tynamo", "Eelektrik",
    "Elgyem", "Litwick", "Lampent", "Axew", "Fraxure", "Cubchoo",
    "Shelmet", "Mienfoo", "Golett", "Pawniard", "Vullaby", "Deino",
    "Zweilous", "Larvesta",

    # ============================================================
    # Gen 6 — pre-evolutions
    # ============================================================
    "Chespin", "Quilladin", "Fennekin", "Braixen", "Froakie",
    "Frogadier", "Bunnelby", "Fletchling", "Fletchinder", "Scatterbug",
    "Spewpa", "Litleo", "Flabébé", "Floette", "Skiddo", "Pancham",
    "Espurr", "Honedge", "Doublade", "Spritzee", "Swirlix", "Inkay",
    "Binacle", "Skrelp", "Clauncher", "Helioptile", "Tyrunt", "Amaura",
    "Goomy", "Sliggoo", "Bergmite", "Phantump", "Pumpkaboo",
}


def is_final_evolution(name: str) -> bool:
    """Return True iff `name` is not in the non-finals lookup."""
    return name not in NON_FINALS


## 2. Data loading (all 6 generations)

Download the CSV, drop Mega / Primal forme rows, keep every generation by default, and tag each Pokemon with `Final_Evolution` using the lookup above.

In [ ]:
from typing import Iterable, Optional

import pandas as pd


POKEMON_CSV_URL = (
    "https://gist.githubusercontent.com/armgilles/"
    "194bcff35001e7eb53a2a8b441e8b2c6/raw/"
    "92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
)

STAT_COLUMNS = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

# Substrings that signal an alternate forme row we want to drop. These
# forms are either introduced in Gen 6 (Megas, Primals) or are variant
# stat lines we don't want to double-count in the base-form analysis.
ALT_FORM_MARKERS = ("Mega ", "Primal ")


def load_pokemon(generations: Optional[Iterable[int]] = None) -> pd.DataFrame:
    """Load the Pokemon CSV, drop alternate forms, optionally filter to
    a subset of generations, and add a boolean `Final_Evolution` column.

    Parameters
    ----------
    generations : iterable of int, or None
        Which generations to keep (e.g. ``[1]`` or ``range(1, 7)``).
        ``None`` (the default) keeps every generation in the CSV.

    Returns
    -------
    pd.DataFrame with the original columns plus `Final_Evolution`.
    """
    df = pd.read_csv(POKEMON_CSV_URL)

    # Drop alternate-forme rows. The CSV labels Mega/Primal forms under
    # the original Pokemon's generation, which inflates those cohorts
    # and distorts the "one row per species" structure the PCA assumes.
    mask_alt = df["Name"].apply(
        lambda n: any(marker in n for marker in ALT_FORM_MARKERS)
    )
    df = df[~mask_alt].reset_index(drop=True)

    if generations is not None:
        df = df[df["Generation"].isin(list(generations))].reset_index(drop=True)

    df["Final_Evolution"] = ~df["Name"].isin(NON_FINALS)
    return df


In [ ]:
pokemon = load_pokemon()
print(f"Loaded {len(pokemon)} Pokemon across all generations.")
print(pokemon.groupby('Generation').size().rename('count'))
print(f"Finals: {pokemon['Final_Evolution'].sum()}  |  Legendaries: {pokemon['Legendary'].sum()}")
pokemon.head()

## 3. PCA with pinned sign conventions

Standardize the 6 stats, fit a 3-component PCA, and flip PC signs so that:

- **PC1** → larger means *more overall power*,
- **PC2** → larger means *more defensive / bulky*,
- **PC3** → larger means *leans physical over special*.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler



def _apply_sign_convention(components_: np.ndarray, projected: np.ndarray):
    """Flip PC signs in-place-ish so the axes read the expected direction."""
    loadings = components_.copy()
    projected = projected.copy()

    # PC1: total power — sum of loadings should be positive.
    if loadings[0].sum() < 0:
        loadings[0] *= -1
        projected[:, 0] *= -1

    # PC2: bulk-vs-speed — HP + Sp. Def loading should be positive.
    if len(loadings) > 1:
        hp_idx = STAT_COLUMNS.index("HP")
        spd_idx = STAT_COLUMNS.index("Sp. Def")
        if loadings[1][hp_idx] + loadings[1][spd_idx] < 0:
            loadings[1] *= -1
            projected[:, 1] *= -1

    # PC3: physical-vs-special — Attack loading should be positive.
    if len(loadings) > 2:
        atk_idx = STAT_COLUMNS.index("Attack")
        if loadings[2][atk_idx] < 0:
            loadings[2] *= -1
            projected[:, 2] *= -1

    return loadings, projected


def compute_pca(df: pd.DataFrame, n_components: int = 3):
    """Standardize the stat columns and fit a PCA with pinned signs.

    Returns
    -------
    pca : fitted sklearn.decomposition.PCA
        The original fitted object. Its `components_` attribute is
        overwritten with the sign-adjusted loadings so that downstream
        tools (e.g. biplot) read the same direction.
    scaler : fitted sklearn.preprocessing.StandardScaler
    components : np.ndarray of shape (n_pokemon, n_components)
        Projections with the sign convention applied.
    """
    X = df[STAT_COLUMNS].to_numpy(dtype=float)

    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    pca = PCA(n_components=n_components)
    projected = pca.fit_transform(X_std)

    loadings, projected = _apply_sign_convention(pca.components_, projected)
    pca.components_ = loadings  # overwrite so biplot uses the same signs

    return pca, scaler, projected


def summary(pca: PCA) -> pd.DataFrame:
    """Readable summary: loadings of each stat on each PC + explained
    variance ratio + cumulative variance."""
    loadings = pd.DataFrame(
        pca.components_.T,
        index=STAT_COLUMNS,
        columns=[f"PC{i + 1}" for i in range(pca.n_components_)],
    )
    loadings.loc["Explained variance ratio"] = pca.explained_variance_ratio_
    loadings.loc["Cumulative"] = np.cumsum(pca.explained_variance_ratio_)
    return loadings.round(3)


In [ ]:
pca, scaler, components = compute_pca(pokemon, n_components=3)
summary(pca)

## 4. Shared dark-mode styling

Centralised Plotly layout so every chart has the same look.

In [ ]:
from __future__ import annotations

# Approximate canonical Pokemon type colors, lightly brightened where
# the originals were too muddy to read on dark.
TYPE_COLORS: dict[str, str] = {
    "Bug": "#C6D36A",
    "Dragon": "#9A7DF8",
    "Electric": "#FFD84D",
    "Fighting": "#E44F44",
    "Fire": "#FF8A4C",
    "Flying": "#C7B8FF",
    "Ghost": "#9F7FC4",
    "Grass": "#95DD6A",
    "Ground": "#EAD484",
    "Ice": "#B8E8E8",
    "Normal": "#C8C8A8",
    "Poison": "#C26BCF",
    "Psychic": "#FF7AAB",
    "Rock": "#D1BB5A",
    "Water": "#6FA8FF",
    "Fairy": "#FFB1C6",
    "Steel": "#D4D4E4",
    "Dark": "#9A8374",
}

# Color scheme for the overall chrome.
BG_COLOR = "rgb(17, 20, 32)"          # plot + paper background
GRID_COLOR = "rgba(255, 255, 255, 0.08)"
ZERO_LINE_COLOR = "rgba(255, 255, 255, 0.18)"
FONT_COLOR = "rgb(220, 225, 240)"
TITLE_COLOR = "rgb(240, 240, 255)"
ACCENT = "#ff4d6d"                    # crimson used for highlights
DIM_POINT = "rgba(160, 170, 190, 0.35)"


def apply_dark_layout(
    fig,
    *,
    title: str | None = None,
    width: int = 960,
    height: int = 640,
    legend_title: str | None = None,
):
    """Apply the shared dark-mode layout to a Plotly figure.

    This centralises the styling so the individual plotting modules can
    stay focused on the data they are drawing rather than on colors and
    spacing.
    """
    fig.update_layout(
        template="plotly_dark",
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        font=dict(
            family="Inter, 'Helvetica Neue', Arial, sans-serif",
            size=13,
            color=FONT_COLOR,
        ),
        title=dict(
            text=title,
            font=dict(size=18, color=TITLE_COLOR),
            x=0.02,
            xanchor="left",
            y=0.96,
            yanchor="top",
        ),
        width=width,
        height=height,
        margin=dict(l=70, r=40, t=70, b=60),
        legend=dict(
            title=dict(text=legend_title or ""),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="rgba(255, 255, 255, 0.1)",
            borderwidth=1,
        ),
        hoverlabel=dict(
            bgcolor="rgb(30, 35, 50)",
            bordercolor="rgba(255, 255, 255, 0.2)",
            font=dict(family="Inter, sans-serif", size=12, color=FONT_COLOR),
        ),
    )

    # 2D axes — only apply if the figure actually has them.
    if "xaxis" in fig.layout:
        fig.update_xaxes(
            gridcolor=GRID_COLOR,
            zerolinecolor=ZERO_LINE_COLOR,
            zerolinewidth=1,
            linecolor="rgba(255, 255, 255, 0.2)",
            tickcolor="rgba(255, 255, 255, 0.2)",
        )
    if "yaxis" in fig.layout:
        fig.update_yaxes(
            gridcolor=GRID_COLOR,
            zerolinecolor=ZERO_LINE_COLOR,
            zerolinewidth=1,
            linecolor="rgba(255, 255, 255, 0.2)",
            tickcolor="rgba(255, 255, 255, 0.2)",
        )

    # 3D scenes — style all three axes + the scene background.
    if "scene" in fig.layout:
        axis_style = dict(
            backgroundcolor=BG_COLOR,
            gridcolor=GRID_COLOR,
            zerolinecolor=ZERO_LINE_COLOR,
            showbackground=True,
            color=FONT_COLOR,
        )
        fig.update_layout(
            scene=dict(
                xaxis=axis_style,
                yaxis=axis_style,
                zaxis=axis_style,
                bgcolor=BG_COLOR,
            )
        )

    return fig


## 5. Biplot — how to read the axes

Overlay the 6 stat loading vectors on the PC1 × PC2 scatter. The direction of each arrow tells you which way Pokemon with high values in that stat drift in the plot. This is what makes the PCA axes interpretable at a glance.

In [ ]:
import numpy as np
import plotly.graph_objects as go



def plot_biplot(df, pca, components, scale: float | None = None):
    """Build an interactive biplot on PC1 x PC2.

    Parameters
    ----------
    df : pd.DataFrame
        Pokemon table (must include Name, Total).
    pca : fitted PCA
    components : np.ndarray, shape (n, >=2)
    scale : float or None
        How far out to draw the loading arrows. If None, auto-pick a
        scale so the longest arrow reaches ~80% of the point cloud.
    """
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]

    # PC1 x PC2 plane of the loadings (rows of pca.components_).
    loadings = pca.components_[:2].T  # shape (6, 2)

    if scale is None:
        pc_extent = max(
            np.abs(components[:, 0]).max(),
            np.abs(components[:, 1]).max(),
        )
        loading_extent = np.linalg.norm(loadings, axis=1).max()
        scale = 0.8 * pc_extent / max(loading_extent, 1e-9)

    fig = go.Figure()

    # Background scatter: dim grey so the arrows pop.
    fig.add_trace(
        go.Scatter(
            x=plot_df["PC1"],
            y=plot_df["PC2"],
            mode="markers",
            marker=dict(size=5, color=DIM_POINT, opacity=0.5,
                        line=dict(width=0)),
            text=plot_df["Name"],
            hovertemplate="<b>%{text}</b><br>Total=%{customdata}<br>"
                          "PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>",
            customdata=plot_df["Total"],
            name="Pokemon",
            showlegend=False,
        )
    )

    # Loading arrows.
    for stat, (lx, ly) in zip(STAT_COLUMNS, loadings):
        x_end = lx * scale
        y_end = ly * scale
        fig.add_annotation(
            x=x_end, y=y_end, ax=0, ay=0,
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True,
            arrowhead=3, arrowsize=1.2, arrowwidth=2,
            arrowcolor=ACCENT,
        )
        # Label at the arrow tip, slightly offset outward.
        r = np.hypot(x_end, y_end)
        offset_x = x_end + 0.12 * (x_end / r if r else 0)
        offset_y = y_end + 0.12 * (y_end / r if r else 0)
        fig.add_annotation(
            x=offset_x, y=offset_y,
            text=f"<b>{stat}</b>",
            showarrow=False,
            font=dict(size=13, color=ACCENT),
            bgcolor="rgba(17,20,32,0.7)",
            borderpad=2,
        )

    apply_dark_layout(fig, title="PCA biplot — loading vectors on PC1 × PC2")
    fig.update_xaxes(title_text="PC1 — overall power →")
    fig.update_yaxes(title_text="PC2 — bulk vs speed ↑")
    return fig


In [ ]:
plot_biplot(pokemon, pca, components)

## 6. Baseline scatter — every Pokemon, Total-coloured

In [ ]:
import pandas as pd
import plotly.express as px



HOVER_FIELDS = {
    "Type 1": True,
    "Type 2": True,
    "Total": True,
    "Generation": True,
    "Legendary": True,
    "PC1": ":.2f",
    "PC2": ":.2f",
}


def plot_2d(df: pd.DataFrame, components):
    """Return a 2D Plotly figure of Pokemon projected onto PC1 x PC2."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]

    fig = px.scatter(
        plot_df,
        x="PC1",
        y="PC2",
        color="Total",  # a neutral gradient gives the points some life
        color_continuous_scale="Viridis",
        hover_name="Name",
        hover_data=HOVER_FIELDS,
    )
    fig.update_traces(
        marker=dict(size=9, opacity=0.85,
                    line=dict(width=0.5, color="rgba(255,255,255,0.3)")),
    )
    apply_dark_layout(fig, title="Pokemon PCA (2D) — all generations")
    fig.update_coloraxes(colorbar=dict(title="Total stats"))
    return fig


def plot_3d(df: pd.DataFrame, components):
    """Return a 3D Plotly figure of Pokemon projected onto PC1 x PC2 x PC3."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]
    plot_df["PC3"] = components[:, 2]

    fig = px.scatter_3d(
        plot_df,
        x="PC1",
        y="PC2",
        z="PC3",
        color="Total",
        color_continuous_scale="Viridis",
        hover_name="Name",
        hover_data={
            "Type 1": True, "Type 2": True,
            "Total": True, "Generation": True, "Legendary": True,
        },
    )
    fig.update_traces(marker=dict(size=3.5, opacity=0.85))
    apply_dark_layout(fig, title="Pokemon PCA (3D) — all generations", height=720)
    fig.update_coloraxes(colorbar=dict(title="Total stats"))
    return fig


In [ ]:
plot_2d(pokemon, components)

In [ ]:
plot_3d(pokemon, components)

## 7. Coloured by primary type

Types are gameplay categories rather than stat-determining labels, so heavy overlap between types is expected.

In [ ]:
import pandas as pd
import plotly.express as px



HOVER_FIELDS = {
    "Type 1": True,
    "Type 2": True,
    "Total": True,
    "Generation": True,
    "Legendary": True,
}


def plot_2d_by_type(df: pd.DataFrame, components):
    """2D PCA scatter colored by the Pokemon's primary type."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]

    fig = px.scatter(
        plot_df,
        x="PC1",
        y="PC2",
        color="Type 1",
        color_discrete_map=TYPE_COLORS,
        hover_name="Name",
        hover_data=HOVER_FIELDS,
    )
    fig.update_traces(
        marker=dict(size=9, opacity=0.85,
                    line=dict(width=0.5, color="rgba(255,255,255,0.25)")),
    )
    apply_dark_layout(
        fig,
        title="Pokemon PCA (2D) — colored by primary type",
        legend_title="Primary type",
    )
    return fig


def plot_3d_by_type(df: pd.DataFrame, components):
    """3D PCA scatter colored by the Pokemon's primary type."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]
    plot_df["PC3"] = components[:, 2]

    fig = px.scatter_3d(
        plot_df,
        x="PC1",
        y="PC2",
        z="PC3",
        color="Type 1",
        color_discrete_map=TYPE_COLORS,
        hover_name="Name",
        hover_data=HOVER_FIELDS,
    )
    fig.update_traces(marker=dict(size=4, opacity=0.9))
    apply_dark_layout(
        fig,
        title="Pokemon PCA (3D) — colored by primary type",
        legend_title="Primary type",
        height=720,
    )
    return fig


In [ ]:
plot_2d_by_type(pokemon, components)

In [ ]:
plot_3d_by_type(pokemon, components)

## 8. Final evolutions only (PCA re-fit on the subset)

In [ ]:
import pandas as pd
import plotly.express as px



HOVER_FIELDS = {
    "Type 1": True,
    "Type 2": True,
    "Total": True,
    "Generation": True,
    "Legendary": True,
}


def plot_finals_2d(finals_df: pd.DataFrame, components):
    """2D PCA scatter of final evolutions, colored by primary type."""
    plot_df = finals_df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]

    fig = px.scatter(
        plot_df,
        x="PC1",
        y="PC2",
        color="Type 1",
        color_discrete_map=TYPE_COLORS,
        hover_name="Name",
        hover_data=HOVER_FIELDS,
    )
    fig.update_traces(
        marker=dict(size=10, opacity=0.9,
                    line=dict(width=0.5, color="rgba(255,255,255,0.3)")),
    )
    apply_dark_layout(
        fig,
        title="Final evolutions — PCA (2D), colored by primary type",
        legend_title="Primary type",
    )
    return fig


def plot_finals_3d(finals_df: pd.DataFrame, components):
    """3D PCA scatter of final evolutions, colored by primary type."""
    plot_df = finals_df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]
    plot_df["PC3"] = components[:, 2]

    fig = px.scatter_3d(
        plot_df,
        x="PC1",
        y="PC2",
        z="PC3",
        color="Type 1",
        color_discrete_map=TYPE_COLORS,
        hover_name="Name",
        hover_data=HOVER_FIELDS,
    )
    fig.update_traces(marker=dict(size=4.5, opacity=0.9))
    apply_dark_layout(
        fig,
        title="Final evolutions — PCA (3D), colored by primary type",
        legend_title="Primary type",
        height=720,
    )
    return fig


In [ ]:
finals = pokemon[pokemon['Final_Evolution']].reset_index(drop=True)
print(f"{len(finals)} final evolutions.")
finals_pca, _, finals_comp = compute_pca(finals, n_components=3)
summary(finals_pca)

In [ ]:
plot_finals_2d(finals, finals_comp)

In [ ]:
plot_finals_3d(finals, finals_comp)

## 9. Legendaries highlighted on the full population

In [ ]:
import plotly.graph_objects as go



def plot_legendaries_2d(df, components):
    """2D PCA scatter with legendaries highlighted and labeled."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]

    normals = plot_df[~plot_df["Legendary"]]
    legendaries = plot_df[plot_df["Legendary"]]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=normals["PC1"],
            y=normals["PC2"],
            mode="markers",
            marker=dict(size=6, color=DIM_POINT, opacity=0.6,
                        line=dict(width=0, color="rgba(255,255,255,0.1)")),
            text=normals["Name"],
            hovertemplate="<b>%{text}</b><br>PC1=%{x:.2f}<br>"
                          "PC2=%{y:.2f}<extra></extra>",
            name="Non-legendary",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=legendaries["PC1"],
            y=legendaries["PC2"],
            mode="markers+text",
            marker=dict(size=13, color=ACCENT,
                        line=dict(width=1, color="rgba(255,255,255,0.7)")),
            text=legendaries["Name"],
            textposition="top center",
            textfont=dict(size=10, color="rgb(240,240,255)"),
            hovertemplate="<b>%{text}</b><br>PC1=%{x:.2f}<br>"
                          "PC2=%{y:.2f}<extra></extra>",
            name="Legendary",
        )
    )
    apply_dark_layout(fig, title="Pokemon PCA (2D) — legendaries highlighted")
    fig.update_xaxes(title_text="PC1 — overall power →")
    fig.update_yaxes(title_text="PC2 — bulk vs speed ↑")
    return fig


def plot_legendaries_3d(df, components):
    """3D PCA scatter with legendaries highlighted."""
    plot_df = df.copy()
    plot_df["PC1"] = components[:, 0]
    plot_df["PC2"] = components[:, 1]
    plot_df["PC3"] = components[:, 2]

    normals = plot_df[~plot_df["Legendary"]]
    legendaries = plot_df[plot_df["Legendary"]]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=normals["PC1"],
            y=normals["PC2"],
            z=normals["PC3"],
            mode="markers",
            marker=dict(size=2.5, color=DIM_POINT, opacity=0.5),
            text=normals["Name"],
            hovertemplate="<b>%{text}</b><extra></extra>",
            name="Non-legendary",
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=legendaries["PC1"],
            y=legendaries["PC2"],
            z=legendaries["PC3"],
            mode="markers+text",
            marker=dict(size=6, color=ACCENT,
                        line=dict(width=1, color="rgba(255,255,255,0.7)")),
            text=legendaries["Name"],
            textfont=dict(size=10, color="rgb(240,240,255)"),
            hovertemplate="<b>%{text}</b><extra></extra>",
            name="Legendary",
        )
    )
    apply_dark_layout(fig, title="Pokemon PCA (3D) — legendaries highlighted",
                      height=720)
    return fig


In [ ]:
plot_legendaries_2d(pokemon, components)

In [ ]:
plot_legendaries_3d(pokemon, components)